In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadPrintRemoveDtRestrictionAMRDebug\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-Beam-Vein");

In [3]:
BoSSSshell.WorkflowMgm.Init("EE-Beam-Vein");

Project name is set to 'EE-Beam-Vein'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\EE-Beam-Vein'.


In [4]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'C:\Users\miao\Documents\Database\EE-Beam-Vein'.


## Create grid

In [5]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid() { 
        double xLeft = -3;
        double xRight = 8;
        double ySize = 2;
        int kelem = 5;

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, ((int)(xRight - xLeft) * kelem) + 1);
        double[] Ynodes = GenericBlas.Linspace(0, ySize, 2 * kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        grd.EdgeTagNames.Add(3, "velocity_inlet_left");
        grd.EdgeTagNames.Add(4, "pressure_outlet_right");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;

            if(Math.Abs(X[1] - 0) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - ySize) <= 1.0e-8)
                et = 2;
            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
                et = 3;
            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [6]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocity(double[] X, double t) {");
           stw.WriteLine("    double R(double t) {");
           stw.WriteLine("        if(t <= 1) {");
           stw.WriteLine("            return (35 + (-84 + (70 - 20 * t) * t) * t) * t * t * t * t;");
           stw.WriteLine("        } else {");
           stw.WriteLine("            return 1;");
           stw.WriteLine("        }");
           stw.WriteLine("    }");
stw.WriteLine("double d = 0.2;");
stw.WriteLine("double G(double[] X) {");
stw.WriteLine("    if(X[1] <= d)");
stw.WriteLine("        return (1 - ((X[1] - d) / d) * ((X[1] - d) / d));");
stw.WriteLine("    else if(X[1] >= 2 - d)");
stw.WriteLine("        return (1 - ((X[1] - 2 + d) / d) * ((X[1] - 2 + d) / d));");
stw.WriteLine("    else");
stw.WriteLine("        return 1;");
stw.WriteLine("}");
           stw.WriteLine("    double vmax = 1;");
           stw.WriteLine("    return vmax * R(t) * G(X);");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    double h = 0.1;");
           stw.WriteLine("    if(X[1] < (0.9 - h) || X[1] > (1.1 + h)) {");
           stw.WriteLine("        return -(X[0] * X[0]) + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    if(X[1] > 1.0) {");
           stw.WriteLine("        return -((X[0] * X[0]) + (X[1] - (1.1 + h)) * (X[1] - (1.1 + h))) + h * h;");
           stw.WriteLine("    }");
           stw.WriteLine("    return -((X[0] * X[0]) + (X[1] - (0.9 - h)) * (X[1] - (0.9 - h))) + h * h;");
           stw.WriteLine("}");

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocity(){
        return new Formula("BoundaryValues.InletVelocity", true, AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [7]:
public static ZLS_Control Channel( int p = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Beam";
    C.SessionName = "Beam" + BeamDensity + "-Vein";
    //C.ProjectDescription = "Droplet running on pc";
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1.0;
    C.PhysicalParameters.rho_B = 1.0;
    C.PhysicalParameters.mu_A = 0.05;
    C.PhysicalParameters.mu_B = 0.05;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        Lame2 = 1000,
        Viscosity = 0.1
        //Lame2 = 1e-6,
        //Viscosity = 0.05
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid());
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_Zero());
    //C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower");
    C.AddBoundaryValue("wall_upper");
    C.AddBoundaryValue("velocity_inlet_left", "VelocityX#A", BoundaryValueFactory.Get_InletVelocity());
    //C.AddBoundaryValue("velocity_inlet_left", "VelocityX#B", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("pressure_outlet_right");
    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    //C.PhysicalParameters.sliplength = 0;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = true;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = true;
    C.DynamicLoadBalancing_Period = 30;
    C.DynamicLoadBalancing_RedistributeAtStartup = true;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    double dt = 5e-3;
    C.dtMax = dt;
    C.dtMin = dt;
    //C.Endtime = 100;
    C.NoOfTimesteps = 2000;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [8]:
//double[] Density = new double[] {0.1, 1, 10, 100}; 
double[] Density = new double[] {1}; 

In [9]:
foreach(double Den in Density){
    var C_CTRLFILE = Channel(2, 2, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 4;
//    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput(); 
}

Grid Edge Tags changed.
Deploying job Beam1-Vein ... 
Set Database: { Session Count = 33; Grid Count = 317; Path = C:\Users\miao\Documents\Database\EE-Beam-Vein }
Grid is not in database yet...
Grid successfully saved: a098f949-b04d-40b6-86cf-722ffc8439df


Deploying executables and additional files ...
Deployment directory: C:\Users\miao\Documents\Database\Deployment\EE-Beam-Vein-ZwoLevelSetSolver2024May12_160627.757110
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.

Starting external console ...
(You may close the new window at any time, the job will continue.)


In [10]:
wmg.DefaultDatabase.Sessions

#0: EE-Beam-Vein	Beam1-Vein*	05/12/2024 09:12:06	a6733db4...
#1: EE-Beam-Vein	Beam1-Vein*	05/12/2024 09:07:15	0b313ac0...
#2: EE-Beam-Vein	Beam1-Vein*	05/12/2024 08:58:55	bb075699...
#3: EE-Beam-Vein	Beam1-Vein*	05/12/2024 08:52:25	6a502e73...
#4: EE-Beam-Vein	Beam1-Vein*	05/12/2024 01:26:53	d55f9502...
#5: EE-Beam-Vein	Beam1-Vein	05/12/2024 01:24:32	807ee199...
#6: EE-Beam-Vein	Beam1-Vein*	05/12/2024 01:22:52	f9825dd1...
#7: EE-Beam-Vein	Beam1-Vein*	05/12/2024 01:20:12	2b260a44...
#8: EE-Beam-Vein	Beam1-Vein	05/12/2024 01:03:51	e43effcc...
#9: EE-Beam-Vein	Beam1-Vein	05/12/2024 01:00:21	d457f3f2...
#10: EE-Beam-Vein	Beam1-Vein	05/12/2024 00:54:21	443b9af8...
#11: EE-Beam-Vein	Beam1-Vein	05/12/2024 00:51:21	12a8e981...
#12: EE-Beam-Vein	Beam1-Vein	05/12/2024 00:46:51	c1d94d56...
#13: EE-Beam-Vein	Beam1-Vein*	05/12/2024 00:44:41	6137e605...
#14: EE-Beam-Vein	Beam1-Vein*	05/12/2024 00:03:41	2fd219a5...
#15: EE-Beam-Vein	Beam1-Vein	05/12/2024 00:01:31	9f1e4c28...
#16: EE-Beam-Vein	Beam1-V

In [11]:
wmg.DefaultDatabase.Sessions[0].Timesteps.Count

11

In [12]:
var outPath = wmg.DefaultDatabase.Sessions[0].Timesteps.Every(1).Skip(920).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: C:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-Beam-Vein__Beam1-Vein__2ef0c73d-dee6-4611-9428-894f57235c6c


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [15]:
wmg.DefaultDatabase.Sessions

#0: EE-Beam-Vein	Beam1-Vein	05/11/2024 20:20:38	076f47eb...


In [ ]:
var session = wmg.DefaultDatabase.Sessions[0];

In [ ]:
session.Timesteps.Count

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("VelocityY").ProbeAt(0.405, 0),
        t => "VelocityY"
        );

In [ ]:
mydataset.PlotNow()

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(518).Export().WithSupersampling(3).Do();

In [ ]:
var DispY = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementY").Single();

In [ ]:
DispY

In [ ]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("Phi2"),
        t => "DisplacementY"
        );

In [ ]:
mydataset.PlotNow()

## Restart

In [ ]:
wmg.DefaultDatabase.Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = wmg.DefaultDatabase.Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].